# Supplementary Figure (Fig 3) 3 - Full treatment enrichment and ASD negatives

Sub-panels under `Fig2/supp_5/<panel>/`. **S5a** full GSEA across collections, both lineages
(the uncurated version of main Panel D); **S5b** the SFARI / HPO ASD negatives (none clear FDR).
SBGNview pathway overlays are pre-rendered in `dirmult_treatment_results/pathway_plots/` (not
re-built here). Frame: hits converge on translation / mitochondrial / RBP machinery, NOT on the
SFARI gene set.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig3/supp_3"
KB = Path(
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/dirmult_treatment_results"
)
g = pd.read_csv(KB / "enrichment_discovery_top_gsea.csv")
g = g[g.kind == "GSEA"]

In [ ]:
# S5a - full GSEA dotplot across collections. Group boxes moved left + full geneset names
# wrapped to 2 lines (no truncation); "GO BP" (no underscore); RBP GO id relabelled. -log10 padj colour.
import textwrap
from matplotlib.patches import FancyBboxPatch

COLS = [
    ("GO_BP", "GO BP"),
    ("Reactome", "Reactome"),
    ("Hallmark", "Hallmark"),
    ("KEGG", "KEGG"),
    ("RBP", "RBP"),
]
COLLAB = dict(COLS)
recs = []
for col, _ in COLS:
    sub = g[g.collection == col]
    top = sub.groupby("Description")["p.adjust"].min().nsmallest(3).index
    for term in top:
        for lin in ["excitatory", "inhibitory"]:
            row = sub[(sub.Description == term) & (sub.lineage == lin)]
            if len(row):
                r = row.iloc[0]
                recs.append(
                    dict(
                        collection=col,
                        term=term,
                        lineage=lin,
                        padj=r["p.adjust"],
                        NES=r["NES"],
                    )
                )
D = pd.DataFrame(recs)


def _clean(t):
    if t == "RBP_GO0003723":
        return "RNA binding (GO:0003723)"
    return t.replace("HALLMARK_", "").replace("_", " ").lower()


D["clean"] = D.term.map(_clean)


def _build_s3a(figsize, wrapw, lblfs):
    terms, seen = [], set()
    for col, _ in COLS:
        for t in D[D.collection == col].drop_duplicates("term").term:
            if t not in seen:
                terms.append((col, t))
                seen.add(t)
    ypos, y, last, grp_rows = {}, 0.0, None, {}
    for col, t in terms:
        if last is not None and col != last:
            y += 0.7
        ypos[t] = y
        grp_rows.setdefault(col, []).append(y)
        y += 1.0
        last = col
    xmap = {"excitatory": 0.0, "inhibitory": 0.6}
    fig, ax = plt.subplots(figsize=figsize)
    nlp = -np.log10(D.padj.clip(lower=1e-300))
    lo, hi = float(nlp.min()), float(nlp.max())
    sizes = 6 + 24 * (nlp - lo) / (hi - lo + 1e-9)
    sc = ax.scatter(
        [xmap[l] for l in D.lineage],
        [ypos[t] for t in D.term],
        s=sizes,
        c=nlp,
        cmap="OrRd",
        vmin=lo,
        vmax=hi,
        edgecolors="black",
        linewidths=0.3,
        zorder=3,
    )
    labmap = {
        t: textwrap.fill(D.loc[D.term == t, "clean"].iloc[0], wrapw) for t in ypos
    }
    for t in ypos:
        ax.text(
            -0.15,
            ypos[t],
            labmap[t],
            fontsize=lblfs,
            va="center",
            ha="right",
            linespacing=0.9,
        )
    STRIP_X, STRIP_W = -1.78, 0.15
    for col, ys in grp_rows.items():
        y0, y1 = min(ys) - 0.5, max(ys) + 0.5
        ax.add_patch(
            FancyBboxPatch(
                (STRIP_X, y0),
                STRIP_W,
                y1 - y0,
                boxstyle="round,pad=0,rounding_size=0.03",
                transform=ax.transData,
                facecolor="#F0F0F0",
                edgecolor="#cccccc",
                linewidth=0.4,
                clip_on=False,
                zorder=1,
            )
        )
        ax.text(
            STRIP_X + STRIP_W / 2,
            np.mean(ys),
            COLLAB[col],
            fontsize=S.FONT_TICK - 1,
            fontweight="bold",
            color="#555",
            va="center",
            ha="center",
            rotation=90,
        )
    ax.set_yticks([])
    ax.set_xticks([0, 0.6])
    ax.set_xticklabels(["EXC", "INH"], fontsize=S.FONT_LABEL)
    for tk, lin in zip(ax.get_xticklabels(), ["excitatory", "inhibitory"]):
        tk.set_color(S.paper_palette_lineage[lin])
        tk.set_fontweight("bold")
    ax.set_xlim(-1.95, 1.0)
    ax.set_ylim(-0.7, max(ypos.values()) + 0.5)
    ax.invert_yaxis()
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=S.FONT_TICK)
    cb.ax.tick_params(labelsize=S.FONT_TICK - 1)
    cb.outline.set_linewidth(0.3)
    ax.set_title(
        "perturbation GSEA - full machinery convergence", fontsize=S.FONT_ANNOT, pad=4
    )
    fig.subplots_adjust(left=0.30, right=0.87, top=0.96, bottom=0.05)
    return fig


S.save_and_show(
    _build_s3a((S.HALF_W_PX / 300, 1300 / 300), 18, S.FONT_TICK - 0.5),
    "fig_s3a_full_gsea",
    figure=FIG,
)
S.save_and_show(
    _build_s3a((S.HALF_W_IN, S.HALF_H_IN), 18, S.FONT_TICK - 0.5),
    "fig_s3a_full_gsea__half_half",
    figure=FIG,
)
S.save_and_show(
    _build_s3a((S.HALF_W_IN, S.THIRD_H_IN), 16, S.FONT_TICK - 1),
    "fig_s3a_full_gsea__third_half",
    figure=FIG,
)

## S5a - ORA companion of the full-enrichment dotplot (consistent with the paper's ORA)


In [ ]:
# S5a (ORA companion) - full ORA dotplot, mirroring the GSEA panel but with over-representation
# analysis (consistent with the ORA used everywhere else in the paper, e.g. fig3b). Same translation
# / ribosome / Myc machinery convergence; RBP is ORA-nonsignificant (shown, honest).
import textwrap
from matplotlib.patches import FancyBboxPatch

g_ora = pd.read_csv(KB / "enrichment_discovery_top_ora.csv")
g_ora = g_ora[g_ora.kind == "ORA"]
COLS_O = [
    ("GO_BP", "GO BP"),
    ("Reactome", "Reactome"),
    ("Hallmark", "Hallmark"),
    ("KEGG", "KEGG"),
    ("RBP", "RBP"),
]
COLLAB_O = dict(COLS_O)
_full_ora_cache = {}


def _full_ora(lin, col):
    """All tested terms of one lineage and collection (the top table keeps only the 10 best per lineage)."""
    if (lin, col) not in _full_ora_cache:
        _full_ora_cache[lin, col] = pd.read_csv(KB / lin / "enrichment_discovery" / f"ora_{col}.csv")
    return _full_ora_cache[lin, col]


_recs = []
for col, _ in COLS_O:
    sub = g_ora[g_ora.collection == col]
    best = sub.groupby("Description")["p.adjust"].min()
    top = best[best < 0.05].nsmallest(3).index  # only terms significant in at least one lineage
    for term in top:
        for lin in ["excitatory", "inhibitory"]:
            row = sub[(sub.Description == term) & (sub.lineage == lin)]
            if not len(row):  # not among this lineage's top terms: take it from the full table
                row = _full_ora(lin, col)
                row = row[row.Description == term]
            if len(row):  # still missing: term not tested in this lineage
                r = row.iloc[0]
                _recs.append(
                    dict(collection=col, term=term, lineage=lin, padj=r["p.adjust"])
                )
Do = pd.DataFrame(_recs)


def _clean_o(t):
    if t == "RBP_GO0003723":
        return "RNA binding (GO:0003723)"
    return t.replace("HALLMARK_", "").replace("_", " ").lower()


Do["clean"] = Do.term.map(_clean_o)


def _build_ora(figsize, wrapw, lblfs):
    terms, seen = [], set()
    for col, _ in COLS_O:
        for t in Do[Do.collection == col].drop_duplicates("term").term:
            if t not in seen:
                terms.append((col, t))
                seen.add(t)
    ypos, y, last, grp_rows = {}, 0.0, None, {}
    for col, t in terms:
        if last is not None and col != last:
            y += 0.7
        ypos[t] = y
        grp_rows.setdefault(col, []).append(y)
        y += 1.0
        last = col
    xmap = {"excitatory": 0.0, "inhibitory": 0.6}
    fig, ax = plt.subplots(figsize=figsize)
    nlp = -np.log10(Do.padj.clip(lower=1e-300))
    lo, hi = float(nlp.min()), float(nlp.max())
    sizes = 6 + 24 * (nlp - lo) / (hi - lo + 1e-9)
    sig = (Do.padj < 0.05).values
    ax.scatter(  # adjusted P >= 0.05: open circles, as in Fig. 2H
        [xmap[l] for l in Do.lineage[~sig]],
        [ypos[t] for t in Do.term[~sig]],
        s=sizes[~sig],
        facecolors="white",
        edgecolors="black",
        linewidths=0.3,
        zorder=3,
    )
    sc = ax.scatter(
        [xmap[l] for l in Do.lineage[sig]],
        [ypos[t] for t in Do.term[sig]],
        s=sizes[sig],
        c=nlp[sig],
        cmap="OrRd",
        vmin=lo,
        vmax=hi,
        edgecolors="black",
        linewidths=0.3,
        zorder=3,
    )
    labmap = {
        t: textwrap.fill(Do.loc[Do.term == t, "clean"].iloc[0], wrapw) for t in ypos
    }
    for t in ypos:
        ax.text(
            -0.15,
            ypos[t],
            labmap[t],
            fontsize=lblfs,
            va="center",
            ha="right",
            linespacing=0.9,
        )
    STRIP_X, STRIP_W = -1.78, 0.15
    for col, ys in grp_rows.items():
        y0, y1 = min(ys) - 0.5, max(ys) + 0.5
        ax.add_patch(
            FancyBboxPatch(
                (STRIP_X, y0),
                STRIP_W,
                y1 - y0,
                boxstyle="round,pad=0,rounding_size=0.03",
                transform=ax.transData,
                facecolor="#F0F0F0",
                edgecolor="#cccccc",
                linewidth=0.4,
                clip_on=False,
                zorder=1,
            )
        )
        ax.text(
            STRIP_X + STRIP_W / 2,
            np.mean(ys),
            COLLAB_O[col],
            fontsize=S.FONT_TICK - 1,
            fontweight="bold",
            color="#555",
            va="center",
            ha="center",
            rotation=90,
        )
    ax.set_yticks([])
    ax.set_xticks([0, 0.6])
    ax.set_xticklabels(["EXC", "INH"], fontsize=S.FONT_LABEL)
    for tk, lin in zip(ax.get_xticklabels(), ["excitatory", "inhibitory"]):
        tk.set_color(S.paper_palette_lineage[lin])
        tk.set_fontweight("bold")
    ax.set_xlim(-1.95, 1.0)
    ax.set_ylim(-0.7, max(ypos.values()) + 0.5)
    ax.invert_yaxis()
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=S.FONT_TICK)
    cb.ax.tick_params(labelsize=S.FONT_TICK - 1)
    cb.outline.set_linewidth(0.3)
    ax.set_title(
        "perturbation ORA - full machinery convergence", fontsize=S.FONT_ANNOT, pad=4
    )
    fig.subplots_adjust(left=0.30, right=0.87, top=0.96, bottom=0.05)
    return fig


S.save_and_show(
    _build_ora((S.HALF_W_PX / 300, 1300 / 300), 18, S.FONT_TICK - 0.5),
    "fig_s3a_full_ora",
    figure=FIG,
)
S.save_and_show(
    _build_ora((S.HALF_W_IN, S.HALF_H_IN), 23, S.FONT_TICK - 0.5),  # wider wrap: long GO terms
    "fig_s3a_full_ora__half_half",
    figure=FIG,
)
S.save_and_show(
    _build_ora((S.HALF_W_IN, S.THIRD_H_IN), 16, S.FONT_TICK - 1),
    "fig_s3a_full_ora__third_half",
    figure=FIG,
)

## S5a ORA - wide (quarter-height full-width, collections split into two blocks)


In [ ]:
# S5a ORA - WIDE variant: quarter-height FULL-width, collections split into two side-by-side
# blocks (GO BP+Reactome | Hallmark+KEGG+RBP) so it reads horizontally. Reuses Do/COLS_O/COLLAB_O.
import textwrap
from matplotlib.patches import FancyBboxPatch

GROUPS = [
    [("GO_BP", "GO BP"), ("Reactome", "Reactome")],
    [("Hallmark", "Hallmark"), ("KEGG", "KEGG"), ("RBP", "RBP")],
]


def _block_terms(cols):
    terms, seen = [], set()
    for col, _ in cols:
        for t in Do[Do.collection == col].drop_duplicates("term").term:
            if t not in seen:
                terms.append((col, t))
                seen.add(t)
    return terms


def _build_ora_wide(figsize, wrapw=18, lblfs=None):
    lblfs = lblfs if lblfs is not None else (S.FONT_TICK - 0.5)
    nlp_all = -np.log10(Do.padj.clip(lower=1e-300))
    lo, hi = float(nlp_all.min()), float(nlp_all.max())
    gmax = 0
    for cols in GROUPS:
        terms = _block_terms(cols)
        ncoll = len({c for c, _ in cols})
        gmax = max(gmax, len(terms) + (ncoll - 1) * 0.7 - 1)
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        1,
        3,
        width_ratios=[1, 1, 0.04],
        wspace=0.05,
        left=0.02,
        right=0.94,
        top=0.86,
        bottom=0.06,
    )
    axes = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]
    sc = None
    for ax, cols in zip(axes, GROUPS):
        terms = _block_terms(cols)
        ypos, y, last, grp_rows = {}, 0.0, None, {}
        for col, t in terms:
            if last is not None and col != last:
                y += 0.7
            ypos[t] = y
            grp_rows.setdefault(col, []).append(y)
            y += 1.0
            last = col
        Dsub = Do[Do.term.isin(ypos)]
        xmap = {"excitatory": 0.0, "inhibitory": 0.6}
        nlp = -np.log10(Dsub.padj.clip(lower=1e-300))
        sizes = 6 + 24 * (nlp - lo) / (hi - lo + 1e-9)
        sig = (Dsub.padj < 0.05).values
        ax.scatter(  # adjusted P >= 0.05: open circles, as in Fig. 2H
            [xmap[l] for l in Dsub.lineage[~sig]],
            [ypos[t] for t in Dsub.term[~sig]],
            s=sizes.values[~sig],
            facecolors="white",
            edgecolors="black",
            linewidths=0.3,
            zorder=3,
        )
        sc = ax.scatter(
            [xmap[l] for l in Dsub.lineage[sig]],
            [ypos[t] for t in Dsub.term[sig]],
            s=sizes.values[sig],
            c=nlp.values[sig],
            cmap="OrRd",
            vmin=lo,
            vmax=hi,
            edgecolors="black",
            linewidths=0.3,
            zorder=3,
        )
        for t in ypos:
            ax.text(
                -0.15,
                ypos[t],
                textwrap.fill(Do.loc[Do.term == t, "clean"].iloc[0], wrapw),
                fontsize=lblfs,
                va="center",
                ha="right",
                linespacing=0.9,
            )
        SX, SW = -1.78, 0.15
        for col, ys in grp_rows.items():
            y0, y1 = min(ys) - 0.5, max(ys) + 0.5
            ax.add_patch(
                FancyBboxPatch(
                    (SX, y0),
                    SW,
                    y1 - y0,
                    boxstyle="round,pad=0,rounding_size=0.03",
                    transform=ax.transData,
                    facecolor="#F0F0F0",
                    edgecolor="#cccccc",
                    linewidth=0.4,
                    clip_on=False,
                    zorder=1,
                )
            )
            ax.text(
                SX + SW / 2,
                np.mean(ys),
                COLLAB_O[col],
                fontsize=S.FONT_TICK - 1,
                fontweight="bold",
                color="#555",
                va="center",
                ha="center",
                rotation=90,
            )
        ax.set_yticks([])
        ax.set_xticks([0, 0.6])
        ax.set_xticklabels(["EXC", "INH"], fontsize=S.FONT_LABEL)
        for tk, lin in zip(ax.get_xticklabels(), ["excitatory", "inhibitory"]):
            tk.set_color(S.paper_palette_lineage[lin])
            tk.set_fontweight("bold")
        ax.set_xlim(-1.95, 1.0)
        ax.set_ylim(-0.7, gmax + 0.5)
        ax.invert_yaxis()
        ax.tick_params(length=0)
        for sp in ax.spines.values():
            sp.set_visible(False)
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(sc, cax=cax)
    cb.set_label(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK - 1)
    cb.outline.set_linewidth(0.3)
    fig.suptitle(
        "perturbation ORA - full machinery convergence", fontsize=S.FONT_ANNOT, y=0.97
    )
    S.save_and_show(fig, "fig_s3a_full_ora__quarter_full", figure=FIG)


_build_ora_wide((S.FULL_W_IN, S.QUARTER_H_IN))

In [ ]:
# S5b - SFARI / ASD negatives: none clear FDR
sp = pd.read_csv(KB / "enrichment_discovery_sfari_panel.csv")
sp["key"] = sp.Description.str.replace("SFARI_", "", regex=False) + " (" + sp.kind + ")"
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 760 / 300))
for lin, off, lab in [("excitatory", -0.18, "EXC"), ("inhibitory", 0.18, "INH")]:
    d = sp[sp.lineage == lin].sort_values("key")
    y = np.arange(len(d))
    ax.barh(
        y + off,
        -np.log10(d["p.adjust"]),
        0.34,
        color=S.paper_palette_lineage[lin],
        label=lab,
    )
ax.axvline(-np.log10(0.05), color="black", ls="--", lw=0.8)
ax.text(
    -np.log10(0.05),
    -0.45,
    "FDR 0.05 ",
    fontsize=S.FONT_ANNOT - 1,
    va="center",
    ha="right",
)
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels(d.sort_values("key").key, fontsize=S.FONT_TICK)
ax.set_xlabel(r"$-\log_{10}$ padj (SFARI gene-set enrichment)", fontsize=S.FONT_LABEL)
ax.set_title(
    f"SFARI / ASD sets NOT enriched in perturbation hits (all padj $\\geq$ {sp['p.adjust'].min():.2f})",
    fontsize=S.FONT_ANNOT - 0.5,
    pad=3,
)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="lower right", title="lineage")
ax.set_xlim(0, max(1.62, -np.log10(sp["p.adjust"].min()) * 1.25))
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=S.FONT_TICK)
fig.subplots_adjust(left=0.30, right=0.96, top=0.90, bottom=0.15)
S.save_and_show(fig, "fig_s3b_sfari_negatives", figure=FIG)